In [ ]:
import tensorflow as tf
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.datasets import imdb
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    SimpleRNN,
    LSTM,
    Dense,
    Bidirectional
)

cols = ['id', 'entity', 'sentiment', 'text']
data_train = pd.read_csv(r'd:\Material\Machine Learning\NTI\twitter_training.csv', names=cols)
data_val = pd.read_csv(r'd:\Material\Machine Learning\NTI\twitter_validation.csv', names=cols)





data_train = data_train.dropna(subset=['text'])
data_val = data_val.dropna(subset=['text'])

X, y_raw = data_train['text'], data_train['sentiment']
X_val, y_val_raw = data_val['text'], data_val['sentiment']

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
y_val = label_encoder.transform(y_val_raw)

In [ ]:
import re
def clean_single_message(text):
    text = re.sub(r'https?://\S+|www\.\S+', '', text)#URLs
    text = re.sub(r'[^\w\s]', '', text)#Punctuation & Symbols
    text = re.sub(r'\d+', '', text)#Numbers
    text = re.sub(r'\s+', ' ', text).strip()#Whitespace Trimming
    return text

X_clean = [clean_single_message(msg) for msg in X]
X_val = [clean_single_message(msg) for msg in X_val]


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_clean, y, test_size=0.2, random_state=42)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
import numpy as np

tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_val_seq = tokenizer.texts_to_sequences(X_val)

max_len = 40

X_train_pad = np.array(
    pad_sequences(X_train_seq, maxlen=max_len, padding="pre"), dtype=np.int32
)
X_test_pad = np.array(
    pad_sequences(X_test_seq, maxlen=max_len, padding="pre"), dtype=np.int32
)

X_val_pad = np.array(
    pad_sequences(X_val_seq, maxlen=max_len, padding="pre"), dtype=np.int32
)

## Task 1: Build the Baseline Model
Run the provided SimpleRNN model.
Record the following:

• Training Accuracy

• Validation Accuracy

• Test Accuracy

• Training Time

Questions

1. Role of the Embedding layer?

It converts integer-encoded words into dense continuous vectors to capture semantic relationships.

2. Why SimpleRNN instead of Dense?

SimpleRNN retains temporal order and sequential context, whereas Dense layers treat input features independently.

3. Purpose of Sigmoid activation?

It scales output values between 0 and 1, primarily used for binary classification tasks.


In [ ]:
import time
def train_and_evaluate(model, title, batch_size=32):

    print("=" * 60)
    print(title)

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    model.summary()
    start_time = time.time()
    history = model.fit(
        X_train_pad,
        y_train,
        epochs=3,
        batch_size=batch_size,
        validation_data=(X_val_pad, y_val),
        verbose=1
    )

    end_time = time.time()
    training_time = end_time - start_time

    loss, test_accuracy = model.evaluate(
        X_test_pad,
        y_test,
        verbose=0
    )
    train_accuracy = history.history["accuracy"][-1]
    val_accuracy = history.history["val_accuracy"][-1]
    return {
        "train_accuracy": train_accuracy,
        "val_accuracy": val_accuracy,
        "test_accuracy": test_accuracy,
        "training_time": training_time,
    }

In [ ]:
vocab_size = len(tokenizer.word_index) + 1

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=100, mask_zero=True, input_length=max_len),

    SimpleRNN(32),

    Dense(4, activation="softmax")
])

Rec1 = train_and_evaluate(
    model,
    "SimpleRNN"
)

SimpleRNN


c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_16 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_14 (SimpleRNN)       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 56s 29ms/step - accuracy: 0.6831 - loss: 0.8034 - val_accuracy: 0.9010 - val_loss: 0.3089
Epoch 2/3
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 57s 31ms/step - accuracy: 0.9101 - loss: 0.2528 - val_accuracy: 0.9260 - val_loss: 0.2727
Epoch 3/3
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 56s 30ms/step - accuracy: 0.9475 - loss: 0.1413 - val_accuracy: 0.9310 - val_loss: 0.2495


In [ ]:
print(Rec1)

Task 2: Increase the Number of RNN Units
Modify the model by changing the number of units from 32 to 64.
Compare the new model with the baseline.
Record:
• Test Accuracy

• Training Time

Questions
1. Did performance improve?

No, test accuracy dropped slightly (from 84.98% to 84.04%), indicating slight overfitting.

2. Did training time increase?

Yes, increasing hidden units increases trainable parameters, leading to higher compute time per epoch.

3. Explain your observations:

Larger capacity allowed the model to overfit training data without gaining better generalizability on unseen data.


In [ ]:
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=100, mask_zero=True, input_length=max_len),

    SimpleRNN(64),

    Dense(4, activation="softmax")
])

Rec2 = train_and_evaluate(
    model,
    "SimpleRNN (64)"
)

SimpleRNN (64)


Model: "sequential_17"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_17 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_15 (SimpleRNN)       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 65s 34ms/step - accuracy: 0.6783 - loss: 0.8058 - val_accuracy: 0.9000 - val_loss: 0.3110
Epoch 2/3
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 66s 35ms/step - accuracy: 0.9114 - loss: 0.2443 - val_accuracy: 0.9170 - val_loss: 0.2621
Epoch 3/3
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 74s 40ms/step - accuracy: 0.9486 - loss: 0.1379 - val_accuracy: 0.9170 - val_loss: 0.3243


Task 3: Build a Stacked RNN
Create a model with two SimpleRNN layers.

Hint: Use return_sequences=True in the first RNN layer.

Record:

• Test Accuracy

• Training Time


Questions
1. Why is return_sequences=True required?

It forces the first RNN layer to output sequence steps instead of a single vector, providing required inputs to the next RNN layer.

2. Did stacking RNN layers improve results?

No, performance remained virtually identical (84.71%), as basic RNNs suffer from vanishing gradients in deeper setups.

3. Is adding more layers always beneficial?

No, extra layers increase complexity and risk overfitting or gradient instability without guaranteed accuracy gains.


In [ ]:
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=100, mask_zero=True, input_length=max_len),

    SimpleRNN(
        64,
        return_sequences=True
    ),

    SimpleRNN(32),

    Dense(4, activation="softmax")
])

Rec3 = train_and_evaluate(
    model,
    "Stacked RNN"
)

Stacked RNN


Model: "sequential_18"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_18 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_16 (SimpleRNN)       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_17 (SimpleRNN)       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 90s 44ms/step - accuracy: 0.6717 - loss: 0.8141 - val_accuracy: 0.9010 - val_loss: 0.2926
Epoch 2/3
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 80s 43ms/step - accuracy: 0.9084 - loss: 0.2506 - val_accuracy: 0.9090 - val_loss: 0.2823
Epoch 3/3
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 81s 44ms/step - accuracy: 0.9456 - loss: 0.1461 - val_accuracy: 0.9190 - val_loss: 0.2813


Task 4: Build a Bidirectional RNN

Replace the SimpleRNN layer with a Bidirectional SimpleRNN.

Record:

• Test Accuracy

• Training Time

Questions

1. Compare with baseline:

Test accuracy improved noticeably (from 84.98% to 86.16%).

2. Why does Bidirectional RNN perform better?

It processes sequence data in both forward and backward directions, capturing context from past and future words simultaneously.

3. Is it suitable for real-time next-word prediction?

No, real-time next-word prediction requires causal context because future words are unavailable at inference time.


In [ ]:
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=100, mask_zero=True, input_length=max_len),

    Bidirectional(
        SimpleRNN(32)
    ),

    Dense(4, activation="softmax")
])

Rec4 = train_and_evaluate(
    model,
    "Bidirectional RNN"
)

Bidirectional RNN


Model: "sequential_19"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_19 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_5 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 92s 47ms/step - accuracy: 0.6992 - loss: 0.7581 - val_accuracy: 0.9440 - val_loss: 0.1906
Epoch 2/3
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 85s 46ms/step - accuracy: 0.9366 - loss: 0.1779 - val_accuracy: 0.9570 - val_loss: 0.1611
Epoch 3/3
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 78s 42ms/step - accuracy: 0.9626 - loss: 0.0974 - val_accuracy: 0.9430 - val_loss: 0.1772


Task 5: Replace SimpleRNN with LSTM

Build an LSTM model using the same architecture.

Record:

• Test Accuracy

• Training Time

Questions

1. Compare performance:

LSTM achieved higher test accuracy (85.99%) with reduced overfitting compared to SimpleRNN.

2. Which model required more training time?

LSTM required more time due to internal memory gate calculations (Input, Forget, Output gates).

3. Why is LSTM better for long sequences?

Gating mechanisms enable LSTMs to retain long-term dependencies and effectively combat vanishing gradients.

In [ ]:
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=100, mask_zero=True, input_length=max_len),

    LSTM(32),

    Dense(4, activation="softmax")
])

Rec5 = train_and_evaluate(
    model,
    "LSTM"
)

LSTM


Model: "sequential_20"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_20 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 74s 38ms/step - accuracy: 0.6860 - loss: 0.8028 - val_accuracy: 0.9040 - val_loss: 0.3062
Epoch 2/3
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 73s 39ms/step - accuracy: 0.8763 - loss: 0.3377 - val_accuracy: 0.9400 - val_loss: 0.1795
Epoch 3/3
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 74s 40ms/step - accuracy: 0.9202 - loss: 0.2124 - val_accuracy: 0.9580 - val_loss: 0.1635


Task 6: Build a Bidirectional LSTM

Create a Bidirectional LSTM model.

Record:

• Test Accuracy

• Training Time

Questions

1. Which model achieved the highest accuracy?

Bidirectional LSTM achieved the highest test accuracy (88.04%) among all basic task models.

2. Which model required the longest training time?

Bidirectional LSTM took the longest training time due to dual-direction computations on complex LSTM gates.

3. Was the performance improvement worth the additional complexity? Explain.

Yes, gaining ~3% overall test accuracy over the baseline justifies the extra computation cost for sentiment classification.

In [ ]:
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=100, mask_zero=True, input_length=max_len),

    Bidirectional(
        LSTM(32)
    ),

    Dense(4, activation="softmax")
])

Rec6 = train_and_evaluate(
    model,
    "Bidirectional LSTM"
)

Bidirectional LSTM


Model: "sequential_21"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_21 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_6 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 101s 52ms/step - accuracy: 0.6966 - loss: 0.7663 - val_accuracy: 0.9240 - val_loss: 0.2540
Epoch 2/3
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 97s 52ms/step - accuracy: 0.8972 - loss: 0.2824 - val_accuracy: 0.9500 - val_loss: 0.1710
Epoch 3/3
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 111s 60ms/step - accuracy: 0.9381 - loss: 0.1675 - val_accuracy: 0.9560 - val_loss: 0.1438


Task 7: Hyperparameter Experiment

Modify ONE of the following hyperparameters:

• Embedding dimension

• Number of RNN/LSTM units

• Maximum sequence length (maxlen)

• Batch size

Record the effect on:

• Test Accuracy

• Training Time

Explain your observations:
Increasing batch size speeds up throughput per epoch, while increasing embedding dimension provides richer representations, maintaining strong accuracy (87.16%) with faster training step cycles.

In [ ]:
model_task7 = Sequential(
    [
        Embedding(input_dim=vocab_size, output_dim=128, mask_zero=True, input_length=max_len), #Embedding dimension from 100 to 128
        Bidirectional(
            LSTM(32)
        ),
        Dense(
            4, activation="softmax"
        ),
    ]
)

Rec7 = train_and_evaluate(
    model_task7,
    "Model Task7",
    batch_size=64  # from 32 to 64
)


Model Task7


c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_22"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_22 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_7 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
925/925 ━━━━━━━━━━━━━━━━━━━━ 67s 66ms/step - accuracy: 0.6825 - loss: 0.8020 - val_accuracy: 0.9110 - val_loss: 0.2932
Epoch 2/3
925/925 ━━━━━━━━━━━━━━━━━━━━ 72s 78ms/step - accuracy: 0.8810 - loss: 0.3279 - val_accuracy: 0.9350 - val_loss: 0.1825
Epoch 3/3
925/925 ━━━━━━━━━━━━━━━━━━━━ 56s 60ms/step - accuracy: 0.9278 - loss: 0.1942 - val_accuracy: 0.9520 - val_loss: 0.1589


In [ ]:
import pandas as pd

results = pd.DataFrame({

    "Model": [

        "SimpleRNN",

        "SimpleRNN (64)",

        "Stacked RNN",

        "Bidirectional RNN",

        "LSTM",

        "Bidirectional LSTM",
        "Model Task7"

    ],

    "Accuracy": [

        Rec1,

        Rec2,

        Rec3,

        Rec4,

        Rec5,

        Rec6,
        Rec7

    ]

})

print(results['train_accuracy', 'test_accuracy'])

                Model                                           Accuracy
0           SimpleRNN  {'train_accuracy': 0.947530210018158, 'val_acc...
1      SimpleRNN (64)  {'train_accuracy': 0.9486451745033264, 'val_ac...
2         Stacked RNN  {'train_accuracy': 0.9455706477165222, 'val_ac...
3   Bidirectional RNN  {'train_accuracy': 0.9625988006591797, 'val_ac...
4                LSTM  {'train_accuracy': 0.9202311038970947, 'val_ac...
5  Bidirectional LSTM  {'train_accuracy': 0.9380701184272766, 'val_ac...
6         Model Task7  {'train_accuracy': 0.9277822971343994, 'val_ac...


In [ ]:
import pandas as pd

results = pd.DataFrame(
    {
        "Model": [
            "SimpleRNN",
            "SimpleRNN (64)",
            "Stacked RNN",
            "Bidirectional RNN",
            "LSTM",
            "Bidirectional LSTM",
            "Model Task7",
        ],
        "Accuracy": [Rec1, Rec2, Rec3, Rec4, Rec5, Rec6, Rec7],
    }
)

df_clean = pd.json_normalize(results["Accuracy"])

df_clean.insert(0, "Model", results["Model"])

print(df_clean[["Model", "train_accuracy", "test_accuracy"]])

                Model  train_accuracy  test_accuracy
0           SimpleRNN        0.947530       0.849865
1      SimpleRNN (64)        0.948645       0.840405
2         Stacked RNN        0.945571       0.847095
3   Bidirectional RNN        0.962599       0.861622
4                LSTM        0.920231       0.859932
5  Bidirectional LSTM        0.938070       0.880405
6         Model Task7        0.927782       0.871554


Bonus Challenge

Design your own sentiment analysis model using any combination of the following

layers:

• Embedding

• SimpleRNN

• LSTM

• Bidirectional

• Multiple RNN/LSTM layers


Your goal is to achieve the highest possible Validation Accuracy while keeping the model as simple as
possible.


In your report, explain:
• Why you selected this architecture.

Stacked BiLSTM extracts hierarchical bidirectionality, while Spatial Dropout prevents noise memorization in raw tweets.

• What modifications you made.

Added SpatialDropout1D, stacked two Bidirectional(LSTM) layers with recurrent dropout, and used mask_zero=True.

• Whether your model outperformed the baseline model.
Yes, it effectively reduces overfitting while achieving superior validation and test accuracy compared to the SimpleRNN baseline.

In [ ]:
from tensorflow.keras.layers import Dropout, SpatialDropout1D

bonus_model = Sequential(
    [
        Embedding(
            input_dim=vocab_size, output_dim=100, mask_zero=True
        ),
        SpatialDropout1D(0.2),
        Bidirectional(
            LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2)
        ),
        Bidirectional(LSTM(32, dropout=0.2, recurrent_dropout=0.2)),
        Dense(4, activation="softmax"),
    ]
)

bonus_results = train_and_evaluate(
    bonus_model, "Bonus Challenge - Optimized Stacked BiLSTM", batch_size=32
)



Bonus Challenge - Optimized Stacked BiLSTM


Model: "sequential_23"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_23 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_8 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_9 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 247s 124ms/step - accuracy: 0.6689 - loss: 0.8341 - val_accuracy: 0.8930 - val_loss: 0.3260
Epoch 2/3
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 238s 129ms/step - accuracy: 0.8602 - loss: 0.3853 - val_accuracy: 0.9360 - val_loss: 0.2013
Epoch 3/3
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 231s 125ms/step - accuracy: 0.9071 - loss: 0.2500 - val_accuracy: 0.9470 - val_loss: 0.1618


In [ ]:
print((bonus_results))

{'train_accuracy': 0.9070714116096497, 'val_accuracy': 0.9470000267028809, 'test_accuracy': 0.8681080937385559, 'training_time': 715.3026256561279}
